# INAI 6986 — Assignment 4 / Milestone 2
## Model Testing, Validation & Refinement

**Project:** Lightweight On-Premises AI-Assisted IT Work Triage for Generalist State Agency IT Teams

This notebook performs the **one-time protected Test evaluation** of the model frozen in Assignment 3.

### Frozen evidence
- Selected model: **TF-IDF + Multinomial Naive Bayes**
- Expected model file: `assignment3_selected_model_naive_bayes.joblib`
- Expected model SHA-256: `e7e440cacfc5926e4eab482bc9baad188eebedf2dbab22b21b3d49fcffac4239`
- Frozen Test file: `Williams_INAI6986_Frozen_Test_Set_v1_0.csv`
- Expected Test CSV SHA-256: `495ec57c87f8bdc2fa77571621e90c882be485379088249f162601eac71fda2e`
- Business threshold: **Escalation recall >= 90%**, meaning at least **18 of 20 escalation cases**
- False-positive safeguard locked before Test: **Escalation precision >= 80%**

### Evaluation rule
The Test set is evaluated once. The model is **not retrained or retuned** after Test exposure. Any misses are analyzed and reported honestly.


## 1. Setup

Run the installation/import cells first. Then upload the **exact frozen model artifact from Assignment 3** and the frozen Test CSV.

If you no longer have the model artifact, return to the executed Assignment 3 Colab notebook and download:

`assignment3_selected_model_naive_bayes.joblib`

Do **not** retrain or replace the model inside this Assignment 4 notebook.


In [ ]:
!pip -q install pandas scikit-learn joblib matplotlib


In [ ]:
import os
import re
import json
import time
import hashlib
import string
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score,
    confusion_matrix, classification_report
)

MODEL_FILE = "assignment3_selected_model_naive_bayes.joblib"
TEST_FILE = "Williams_INAI6986_Frozen_Test_Set_v1_0.csv"

EXPECTED_MODEL_SHA256 = "e7e440cacfc5926e4eab482bc9baad188eebedf2dbab22b21b3d49fcffac4239"
EXPECTED_TEST_SHA256 = "495ec57c87f8bdc2fa77571621e90c882be485379088249f162601eac71fda2e"

BUSINESS_RECALL_THRESHOLD = 0.90
FALSE_POSITIVE_PRECISION_SAFEGUARD = 0.80

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not (os.path.exists(MODEL_FILE) and os.path.exists(TEST_FILE)):
    try:
        from google.colab import files
        print("Upload BOTH files:")
        print("  1.", MODEL_FILE)
        print("  2.", TEST_FILE)
        files.upload()
    except Exception:
        pass

assert os.path.exists(MODEL_FILE), f"Missing {MODEL_FILE}"
assert os.path.exists(TEST_FILE), f"Missing {TEST_FILE}"

model_hash = sha256_file(MODEL_FILE)
test_hash = sha256_file(TEST_FILE)

print("Model SHA-256:", model_hash)
print("Test  SHA-256:", test_hash)

assert model_hash == EXPECTED_MODEL_SHA256, (
    "STOP: Model hash does not match the frozen Assignment 3 artifact. "
    "Do not run Test with a different or retrained model."
)
assert test_hash == EXPECTED_TEST_SHA256, (
    "STOP: Test file hash does not match the frozen protected Test artifact."
)

print("\nPASS: Both frozen artifacts match their recorded SHA-256 hashes.")


## 2. Inspect Test Metadata Without Scoring

This cell verifies the Test structure only. It does not fit or tune anything.

Expected:
- 40 cases
- 20 Generalist / 20 Escalate
- 10 cases per domain
- four domains


In [ ]:
test_df = pd.read_csv(TEST_FILE)

required = {"scenario_id", "domain", "scenario_text", "escalation_label"}
missing = required - set(test_df.columns)
assert not missing, f"Missing required columns: {missing}"

assert len(test_df) == 40, f"Expected 40 Test cases; found {len(test_df)}"
assert test_df["escalation_label"].value_counts().to_dict() == {0: 20, 1: 20}

domain_counts = test_df["domain"].value_counts().sort_index()
assert len(domain_counts) == 4
assert all(domain_counts == 10), f"Expected 10 cases/domain; found {domain_counts.to_dict()}"

print("Test rows:", len(test_df))
print("Labels:", test_df["escalation_label"].value_counts().sort_index().to_dict())
print("Domains:")
print(domain_counts)


## 3. Frozen Baseline Definition

Assignment 4 compares the frozen ML model with the fixed Assignment 3 keyword/risk-rule baseline on the same Test split.

The baseline below is copied from Assignment 3 and must not be edited after Test evaluation begins.


In [ ]:
BASELINE_RISK_PATTERNS = [
    r"\bunsupported\b",
    r"\bend[- ]of[- ]support\b",
    r"\bdisable\b.*\bmfa\b",
    r"\bexempt\b.*\bmfa\b",
    r"\bpermanent\b.*\badmin",
    r"\bprivileged\b.*\baccess\b",
    r"\bpublic internet\b",
    r"\bexpose\b.*\bpublic\b",
    r"\bremove\b.*\bencryption\b",
    r"\bdisable\b.*\bencryption\b",
    r"\bshorten\b.*\bretention\b",
    r"\breduce\b.*\bretention\b",
    r"\bchange\b.*\brpo\b",
    r"\bchange\b.*\brto\b",
    r"\brecovery point objective\b",
    r"\brecovery time objective\b",
    r"\bpolicy exception\b",
    r"\bpolicy waiver\b",
    r"\bsecurity exception\b",
    r"\bbypass\b.*\bpolicy\b",
    r"\bnot included in the executed contract\b",
    r"\boutside\b.*\bcontract\b",
    r"\badditional implementation work\b",
    r"\bdiagnostic copies\b.*\bagency data\b",
    r"\bstop backing up\b",
    r"\bturn off centralized logging\b",
    r"\bexclude\b.*\bendpoint protection\b",
]

baseline_regex = re.compile("|".join(BASELINE_RISK_PATTERNS), flags=re.IGNORECASE)

def baseline_predict(texts):
    return np.array(
        [1 if baseline_regex.search(str(t)) else 0 for t in texts],
        dtype=int
    )

print("Frozen baseline patterns:", len(BASELINE_RISK_PATTERNS))


## 4. ONE-TIME TEST GATE

**Before changing the value below:**
1. Confirm both SHA-256 checks passed.
2. Confirm the baseline definition is frozen.
3. Confirm the selected model will not be changed after seeing Test results.
4. Confirm the business threshold (90% recall) and safeguard (80% precision) are locked.

Then change:

`CONFIRM_ONE_TIME_TEST = False`

to:

`CONFIRM_ONE_TIME_TEST = True`

Run the cell once and proceed directly to the evaluation cell.


In [ ]:
CONFIRM_ONE_TIME_TEST = False

assert CONFIRM_ONE_TIME_TEST is True, (
    "Protected Test evaluation is locked. "
    "Set CONFIRM_ONE_TIME_TEST = True only when you are ready for the one-time Test run."
)

print("ONE-TIME TEST GATE OPEN.")


## 5. Primary Protected Test Evaluation

This is the **one-time frozen Test run**.

Do not change the model, TF-IDF settings, decision rule, baseline patterns, business threshold, or safeguard after this cell is executed.


In [ ]:
LOCK_FILE = "ASSIGNMENT4_PRIMARY_TEST_COMPLETE.lock"

if os.path.exists(LOCK_FILE):
    raise RuntimeError(
        "Primary Test evaluation was already completed in this notebook session. "
        "Do not rerun it for tuning."
    )

model = joblib.load(MODEL_FILE)

X_test = test_df["scenario_text"].astype(str)
y_test = test_df["escalation_label"].astype(int)

def evaluate(name, y_true, y_pred, runtime_sec):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "System": name,
        "Recall_Escalation": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Precision_Escalation": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1_Escalation": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "False_Negatives": int(fn),
        "False_Positives": int(fp),
        "Escalation_Rate": float(np.mean(y_pred)),
        "Runtime_sec": float(runtime_sec),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

start = time.perf_counter()
baseline_pred = baseline_predict(X_test)
baseline_runtime = time.perf_counter() - start

start = time.perf_counter()
model_pred = model.predict(X_test)
model_runtime = time.perf_counter() - start

primary_results = pd.DataFrame([
    evaluate("Keyword/Risk Baseline", y_test, baseline_pred, baseline_runtime),
    evaluate("Frozen Naive Bayes", y_test, model_pred, model_runtime),
])

with open(LOCK_FILE, "w") as f:
    f.write(datetime.now(timezone.utc).isoformat())

display(primary_results[[
    "System", "Recall_Escalation", "Precision_Escalation", "F1_Escalation",
    "False_Negatives", "False_Positives", "Accuracy", "Escalation_Rate", "Runtime_sec"
]])

model_row = primary_results[primary_results["System"] == "Frozen Naive Bayes"].iloc[0]

recall_met = model_row["Recall_Escalation"] >= BUSINESS_RECALL_THRESHOLD
precision_met = model_row["Precision_Escalation"] >= FALSE_POSITIVE_PRECISION_SAFEGUARD

print("\nBusiness interpretation")
print("-----------------------")
print(f"Escalations caught: {int(model_row['TP'])} of 20")
print(f"Escalations missed: {int(model_row['FN'])} of 20")
print(f"Routine cases incorrectly escalated: {int(model_row['FP'])} of 20")
print(f"Recall threshold >= {BUSINESS_RECALL_THRESHOLD:.0%}: {'MET' if recall_met else 'NOT MET'}")
print(f"Precision safeguard >= {FALSE_POSITIVE_PRECISION_SAFEGUARD:.0%}: {'MET' if precision_met else 'NOT MET'}")


## 6. Confusion Matrix and Misclassified Cases


In [ ]:
cm = confusion_matrix(y_test, model_pred, labels=[0,1])

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm)
ax.set_xticks([0,1], labels=["Generalist (0)", "Escalate (1)"])
ax.set_yticks([0,1], labels=["Generalist (0)", "Escalate (1)"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Frozen Naive Bayes — Protected Test Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center")

plt.tight_layout()
plt.savefig("assignment4_test_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

pred_out = test_df[["scenario_id","domain","scenario_text","escalation_label"]].copy()
pred_out["baseline_prediction"] = baseline_pred
pred_out["model_prediction"] = model_pred
pred_out["correct"] = pred_out["escalation_label"] == pred_out["model_prediction"]

misclassified = pred_out[~pred_out["correct"]].copy()

print("Misclassified cases:", len(misclassified))
display(misclassified)


## 7. Subgroup Performance by Domain

Assignment 4 requires performance broken out across the segments affected by the decision.  
Here, the four operational domains are the primary subgroups.


In [ ]:
domain_rows = []

for domain, grp in pred_out.groupby("domain"):
    yt = grp["escalation_label"].astype(int)
    yp = grp["model_prediction"].astype(int)

    cm_d = confusion_matrix(yt, yp, labels=[0,1])
    tn, fp, fn, tp = cm_d.ravel()

    domain_rows.append({
        "Domain": domain,
        "Cases": len(grp),
        "Escalation_Cases": int((yt == 1).sum()),
        "Recall_Escalation": recall_score(yt, yp, pos_label=1, zero_division=0),
        "Precision_Escalation": precision_score(yt, yp, pos_label=1, zero_division=0),
        "F1_Escalation": f1_score(yt, yp, pos_label=1, zero_division=0),
        "Accuracy": accuracy_score(yt, yp),
        "FN": int(fn),
        "FP": int(fp),
        "TP": int(tp),
        "TN": int(tn),
    })

domain_performance = pd.DataFrame(domain_rows).sort_values("Domain").reset_index(drop=True)
display(domain_performance)


## 8. Named Error Analysis

This section summarizes the actual Test misses by domain and error type.  
The text of each missed case should be reviewed manually when writing Milestone 2 to identify meaningful failure modes (for example: subtle authority boundary, implicit policy conflict, ambiguous wording, or domain-specific terminology).


In [ ]:
if len(misclassified) == 0:
    print("No Test misclassifications were observed.")
else:
    error_summary = (
        misclassified
        .assign(
            Error_Type=np.where(
                (misclassified["escalation_label"] == 1) & (misclassified["model_prediction"] == 0),
                "False Negative — missed escalation",
                "False Positive — unnecessary escalation"
            )
        )
        .groupby(["domain","Error_Type"])
        .size()
        .reset_index(name="Count")
    )
    display(error_summary)

    display(
        misclassified.assign(
            Error_Type=np.where(
                (misclassified["escalation_label"] == 1) & (misclassified["model_prediction"] == 0),
                "False Negative — missed escalation",
                "False Positive — unnecessary escalation"
            )
        )[["scenario_id","domain","Error_Type","scenario_text"]]
    )


## 9. Robustness Checks — Evaluation Only, No Tuning

The frozen model is now tested on realistic input changes **without retraining**.

Two deterministic checks are included:

1. **Formatting noise:** punctuation removed and whitespace normalized. This approximates work-item text entered with inconsistent formatting.
2. **Truncation:** keeps roughly the first 70% of words (minimum 8 words), approximating an incomplete or shortened intake description.

These checks do not change the original protected-Test result and must not be used to retune the model.


In [ ]:
def formatting_noise(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def truncate_text(text, keep_fraction=0.70, min_words=8):
    words = str(text).split()
    if len(words) <= min_words:
        return str(text)
    keep = max(min_words, int(round(len(words) * keep_fraction)))
    return " ".join(words[:keep])

robustness_variants = {
    "Original Protected Test": X_test.tolist(),
    "Formatting Noise": [formatting_noise(x) for x in X_test],
    "70% Truncated Intake": [truncate_text(x) for x in X_test],
}

robustness_rows = []

for variant, texts in robustness_variants.items():
    pred = model.predict(texts)
    row = evaluate(variant, y_test, pred, 0.0)
    robustness_rows.append(row)

robustness = pd.DataFrame(robustness_rows)

original_recall = robustness.loc[
    robustness["System"] == "Original Protected Test", "Recall_Escalation"
].iloc[0]

robustness["Recall_Delta_vs_Original"] = (
    robustness["Recall_Escalation"] - original_recall
)

display(robustness[[
    "System","Recall_Escalation","Recall_Delta_vs_Original",
    "Precision_Escalation","F1_Escalation","False_Negatives",
    "False_Positives","Accuracy"
]])


## 10. Deployment-Readiness Evidence

This cell summarizes the precommitted decision criteria.  
The final deployment recommendation remains a **human decision** and should account for the error analysis, subgroup gaps, robustness behavior, operational controls, and governance requirements—not only the aggregate score.


In [ ]:
model_row = primary_results[primary_results["System"] == "Frozen Naive Bayes"].iloc[0]

readiness = pd.DataFrame([
    {
        "Criterion": "Escalation recall business threshold",
        "Requirement": ">= 90%",
        "Observed": f"{model_row['Recall_Escalation']:.1%}",
        "Status": "MET" if model_row["Recall_Escalation"] >= BUSINESS_RECALL_THRESHOLD else "NOT MET",
    },
    {
        "Criterion": "Escalation precision safeguard",
        "Requirement": ">= 80%",
        "Observed": f"{model_row['Precision_Escalation']:.1%}",
        "Status": "MET" if model_row["Precision_Escalation"] >= FALSE_POSITIVE_PRECISION_SAFEGUARD else "NOT MET",
    },
    {
        "Criterion": "Protected Test integrity",
        "Requirement": "Frozen hashes match",
        "Observed": "Model + Test hashes verified",
        "Status": "MET",
    },
    {
        "Criterion": "Human oversight",
        "Requirement": "Advisory; no autonomous production action",
        "Observed": "Required by project scope",
        "Status": "REQUIRED",
    },
])

display(readiness)


## 11. Save Assignment 4 Evidence

This produces the evidence files needed for the Milestone 2 paper and repository.


In [ ]:
primary_results.to_csv("assignment4_test_metrics.csv", index=False)
pred_out.to_csv("assignment4_test_predictions.csv", index=False)
domain_performance.to_csv("assignment4_domain_performance.csv", index=False)
robustness.to_csv("assignment4_robustness_results.csv", index=False)

run_log = {
    "assignment": "INAI 6986 Assignment 4 / Milestone 2",
    "evaluation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_file": MODEL_FILE,
    "model_sha256": model_hash,
    "test_file": TEST_FILE,
    "test_sha256": test_hash,
    "test_rows": int(len(test_df)),
    "business_recall_threshold": BUSINESS_RECALL_THRESHOLD,
    "precision_safeguard": FALSE_POSITIVE_PRECISION_SAFEGUARD,
    "baseline_pattern_count": len(BASELINE_RISK_PATTERNS),
    "primary_results": primary_results.to_dict(orient="records"),
    "domain_performance": domain_performance.to_dict(orient="records"),
    "robustness_results": robustness.to_dict(orient="records"),
    "misclassified_case_ids": misclassified["scenario_id"].tolist(),
    "post_test_tuning_performed": False,
}

with open("assignment4_run_log.json", "w") as f:
    json.dump(run_log, f, indent=2)

print("Saved:")
for name in [
    "assignment4_test_metrics.csv",
    "assignment4_test_predictions.csv",
    "assignment4_domain_performance.csv",
    "assignment4_robustness_results.csv",
    "assignment4_test_confusion_matrix.png",
    "assignment4_run_log.json",
]:
    print(" -", name)


## 12. Milestone 2 Write-Up Checklist

Use these outputs to complete the official 5–7 page Milestone 2 submission:

1. **Summary of results** — baseline vs frozen model vs 90% business threshold, translated into cases caught/missed.
2. **Experimental setup** — frozen TF-IDF + Naive Bayes pipeline, protected Test procedure, hashes, metrics, and locked precision safeguard.
3. **Iteration narrative** — baseline → LR/SVM/NB Validation comparison → Naive Bayes freeze; include the weak baseline and alternatives that were not selected.
4. **Test results** — same-split comparison table and confusion matrix.
5. **Error & robustness analysis** — named failure modes with counts plus robustness recall deltas.
6. **Subgroup performance** — all four domains with counts and observed gaps.
7. **Limitations & deployment readiness** — synthetic-data validity, wording sensitivity, human review, monitoring, and retraining triggers.
8. **Repository** — notebook plus saved evidence files.
9. **AI Use Disclosure** — 2–6 lines describing ChatGPT/Colab use and what you personally checked.

### Critical rule
Do not alter the frozen model or tune to the protected Test result. Assignment 4 is evaluation evidence, not a second model-selection round.
